# 07 · Tracing and eval

To compare three frameworks, every run has to be measured the same way: the same definition of a token,
an LLM call, a tool call and a second. This notebook connects LangGraph to the shared harness in
`agent_contracts.harness`, measures runs of the research workflow from notebook 05, and scores them with
the shared scoring, which is how the capstone's `run_eval.py` will work.

**You will learn**

- what the shared harness measures, and where each number comes from
- how a LangChain callback feeds model calls into it
- how to run a small, scored evaluation and write it in the shared results format

In [1]:
from agent_contracts.harness import RunMetrics, build_results, verify_model
from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report
from lg_agents.llm import make_llm
from lg_agents.research_flow import build_research_flow, to_report

verify_model()
flow = build_research_flow(make_llm())
questions = {q["id"]: q for q in load_questions()}

## The shared definitions

| Number | Definition | Where it comes from |
|---|---|---|
| LLM calls | every model call in the run | the framework's callbacks or hooks |
| input and output tokens | summed over every model call | the counts Ollama reports with each response |
| tool calls | every attempted call, refused ones included | the shared tools record them (`research.tool_calls()`) |
| latency | wall clock for the whole question | `RunMetrics.measure()` |

Tool calls are counted inside the shared tools, so no framework can count them differently. Model calls
have to come from the framework, because only it sees them; in LangChain that is a **callback**.

In [2]:
import inspect

import lg_agents.metrics

print(inspect.getsource(lg_agents.metrics.HarnessCallback))

class HarnessCallback(BaseCallbackHandler):
    def __init__(self, metrics: RunMetrics) -> None:
        self.metrics = metrics

    def on_llm_end(self, response, **kwargs) -> None:
        for generations in response.generations:
            for generation in generations:
                usage = getattr(getattr(generation, "message", None), "usage_metadata", None) or {}
                self.metrics.add_llm_call(usage.get("input_tokens"), usage.get("output_tokens"))



Callbacks passed in the run's `config` reach every nested model call, including the researcher agent
running inside the workflow's `research` node.

## One measured run

In [3]:
from lg_agents.metrics import HarnessCallback


def run_question(qid: str) -> dict:
    """Run one question through the workflow, measured and scored the shared way."""
    q = questions[qid]
    research.reset()
    metrics = RunMetrics()
    with metrics.measure():
        state = flow.invoke({"question_id": qid, "question": q["question"]},
                            config={"callbacks": [HarnessCallback(metrics)]})
    report = to_report(state)
    scored = score_report(report, q)
    return {"id": qid, "passed": scored["passed"], "scores": scored["scores"], "problems": scored["problems"],
            "report": report.model_dump(), "tool_calls": research.tool_calls(),
            "budget_exhausted": research.budget_exhausted(), **metrics.as_dict(), "error": None}


item = run_question("Q07_deployment")
print({k: item[k] for k in ("passed", "llm_calls", "input_tokens", "output_tokens", "latency_s", "tool_calls")})
print("answer:", item["report"]["answer"])
print("scores:", item["scores"], item["problems"])

{'passed': True, 'llm_calls': 5, 'input_tokens': 2671, 'output_tokens': 295, 'latency_s': 74.406, 'tool_calls': ['web_search']}
answer: Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces as of the end of 2025 [F04].
scores: {'fact_recall': 1.0, 'facts_missing': [], 'contradiction_free': True, 'citations_valid': True} []


Q07 asks how many units Veltara has deployed and in how many provinces. The 2023 article still says
9 provinces; the current figure is 14. Mentioning 9 at all fails `contradiction_free`.

## A small eval

An evaluation is the same thing over a fixed set of cases, scored automatically, so you can re-run it after
every change and see whether things got better. Here are three more questions: a calculation (Q03), a
recall with a distractor article about another company (Q08), and the Series B round (Q04). Each takes a
minute or two on the reference Mac.

In [4]:
items = [item]
for qid in ("Q03_revenue_growth", "Q08_recall", "Q04_series_b"):
    items.append(run_question(qid))

print(f"{'question':20} {'pass':5} {'recall':6} {'calls':5} {'tokens':>7} {'secs':>5}  tools")
for it in items:
    print(f"{it['id']:20} {str(it['passed']):5} {it['scores']['fact_recall']:6} {it['llm_calls']:5} "
          f"{it['input_tokens'] + it['output_tokens']:7} {it['latency_s']:5.0f}  {it['tool_calls']}")
for it in items:
    if it["problems"]:
        print(f"\n{it['id']}: {it['problems']}\n  answer: {it['report']['answer']}")

question             pass  recall calls  tokens  secs  tools
Q07_deployment       True     1.0     5    2966    74  ['web_search']
Q03_revenue_growth   True     1.0     6    4525    74  ['web_search', 'calculator']
Q08_recall           True     1.0     6    5039    94  ['web_search', 'web_search']
Q04_series_b         True     1.0     5    3139    68  ['web_search']


## The shared results format

`build_results` wraps the items in the format every framework writes, and computes the summary the same way
for all of them. The capstone's `run_eval.py` writes it to `results/research/langgraph.json`; this notebook
only prints it, so a tutorial run never overwrites real results.

In [5]:
from importlib.metadata import version

results = build_results("research", "langgraph", version("langgraph"), flow_model := make_llm().model, items)
print("model:", flow_model)
print("summary:", results["summary"])
print("keys per question:", sorted(results["questions"][0]))

model: qwen3-agents
summary: {'passed': 4, 'total': 4, 'total_tokens': 15669, 'total_latency_s': 310.044}
keys per question: ['budget_exhausted', 'error', 'id', 'input_tokens', 'latency_s', 'llm_calls', 'output_tokens', 'passed', 'problems', 'report', 'scores', 'tool_calls']


## Seeing inside a run

Numbers say *how much*; traces say *what happened*. You have already used the simplest trace:
`stream_mode="updates"` (notebooks 03 and 05). LangGraph also offers `stream_mode="debug"` for every event,
and LangSmith, a hosted tracing service, which needs an account and API key and is not used in this
local-only tutorial. Whatever the tool, the traces are what you read when an eval score drops.

## How the others do this

- **CrewAI** publishes events on `crewai_event_bus`; a `BaseEventListener` subclass can collect model
  calls and token usage. Crews also report accumulated usage after a run.
- **Strands** uses hooks: `AfterModelCallEvent` and `AfterToolCallEvent` fire around each call, and an
  agent's result carries `metrics.accumulated_usage`.

All three feed the same `RunMetrics`, and all three count tool calls through the shared tools.

## Takeaways

- Fix the definitions once, in shared code, then connect each framework to them.
- Count what you can outside the framework (tool calls); take only model calls from the framework.
- An eval is a fixed set of cases, scored automatically, re-run after every change.
- That completes the components. The capstones build ShipFast and the research agent in full.